# Complete Synthetic PDW Dataset Generator Demo
This notebook demonstrates the generation of the final synthetic PDW dataset for Part 1.5 of the project. It includes:
1. Interleaved Radars
2. Missing Pulses & False Pulses (Noise & Multipath Echoes)
3. All Secondary Parameters (Pulse Width, Amplitude, Angle of Arrival)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from radar_generator import Environment, StableRadar, JitterRadar, StaggerRadar, SlidingRadar

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Setting up the Complete Environment

In [ ]:
env = Environment()

# 1. Stable PRI with Frequency Hopping
# Setting PW to 1.5us, Amplitude to -65dBm, arriving from 45 degrees
stable_radar = StableRadar(
    radar_id="R_STABLE", 
    pri=1000.0, 
    freq_mean=9000.0, 
    freq_mode="hopping", 
    freq_hopping_values=[9000.0, 9100.0, 9200.0],
    poi=0.70,
    multipath_prob=0.10,
    multipath_delay_mean=1.5,
    pw_mean=1.5,
    amp_mean=-65.0,
    aoa_mean=45.0
)
env.add_radar(stable_radar)

# 2. Jitter PRI with Constant Frequency
# Setting PW to 0.5us, Amplitude to -50dBm (stronger), arriving from 120 degrees
jitter_radar = JitterRadar(
    radar_id="R_JITTER", 
    pri_mean=1500.0, 
    pri_jitter_std=50.0, 
    freq_mean=9500.0,
    poi=0.95,
    pw_mean=0.5,
    amp_mean=-50.0,
    aoa_mean=120.0
)
env.add_radar(jitter_radar)

# 3. Stagger PRI with Frequency Agility
# Setting PW to 2.0us, Amplitude to -75dBm (weaker), arriving from 210 degrees
stagger_radar = StaggerRadar(
    radar_id="R_STAGGER", 
    stagger_pris=[800.0, 1200.0, 1500.0], 
    freq_mean=8500.0, 
    freq_mode="agility", 
    freq_hopping_values=[8500.0, 8600.0, 8700.0, 8800.0],
    poi=1.0,
    pw_mean=2.0,
    amp_mean=-75.0,
    aoa_mean=210.0
)
env.add_radar(stagger_radar)

# Generate 50,000 microseconds (50 ms) of interleaved data
df_interleaved = env.generate(
    duration=50000.0, 
    toa_noise_std=5.0, 
    freq_noise_std=2.0,
    global_drop_rate=0.05,
    noise_rate=1000.0
)

print(f"Total pulses generated: {len(df_interleaved)}")
display(df_interleaved.head(10))

## 2. Visualization

In [ ]:
plt.figure(figsize=(12, 6))

# Plot TOA vs Frequency
sns.scatterplot(data=df_interleaved, x='TOA', y='Frequency', hue='Radar_ID', palette='Paired', s=60, marker='x')

plt.title("Interleaved Radar Environment (TOA vs Frequency)")
plt.xlabel("Time of Arrival (us)")
plt.ylabel("Frequency (MHz)")
plt.grid(True)
plt.legend(title='Radar Emitter', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# Plot TOA vs Measured Delta TOA
plt.plot(df_interleaved['TOA'], df_interleaved['Measured_Delta_TOA'], color='gray', linestyle='-', alpha=0.5, label="Measured Delta TOA")
sns.scatterplot(data=df_interleaved, x='TOA', y='Measured_Delta_TOA', hue='Radar_ID', palette='Paired', s=50)

plt.title("Interleaved Radar Environment (TOA vs Measured Delta TOA)")
plt.xlabel("Time of Arrival (us)")
plt.ylabel("Delta TOA from previous pulse (us)")
plt.grid(True)
plt.legend(title='Ground Truth', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 10))

# Polar plot for Angle of Arrival and Amplitude
ax = plt.subplot(111, projection='polar')

unique_radars = df_interleaved['Radar_ID'].unique()
colors = plt.cm.Paired(np.linspace(0, 1, len(unique_radars)))

for idx, radar_id in enumerate(unique_radars):
    subset = df_interleaved[df_interleaved['Radar_ID'] == radar_id]
    theta = np.deg2rad(subset['AOA'])
    r = subset['Amplitude'] + 100 # Shift for visualization
    ax.scatter(theta, r, label=radar_id, color=colors[idx], alpha=0.7, edgecolors='none')

ax.set_title("Polar Plot: Angle of Arrival (AOA) vs Amplitude", va='bottom')
plt.legend(bbox_to_anchor=(1.1, 1), loc='upper left')
plt.show()